In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
from torch import nn
from transformers import (
    Trainer,
    TrainingArguments,
    LlamaTokenizer,
    LlamaForSequenceClassification,
    TrainerCallback,
    default_data_collator,
)

In [3]:
from accelerate.utils import load_and_quantize_model
from accelerate.utils import BnbQuantizationConfig
from accelerate import init_empty_weights

In [4]:

class train_config:
    def __init__(self):
        self.quantization: bool = False


globalconfig = train_config()
globalconfig.model_id = f"/bime-munin/llama2_hf/llama-2-7b_hf/"

In [5]:
tokenizer = LlamaTokenizer.from_pretrained(f"/bime-munin/llama2_hf/llama-2-7b_hf/")

tokenizer.add_special_tokens({"pad_token": "<pad>"})


1

In [6]:
from accelerate import init_empty_weights
with init_empty_weights():

    model = LlamaForSequenceClassification.from_pretrained(
        globalconfig.model_id,
        # device_map="auto",
        device_map="cpu",
        # load_in_8bit=args.quantization,
        # torch_dtype=torch.float16,
    )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at /bime-munin/llama2_hf/llama-2-7b_hf/ and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
model.config.pad_token_id = tokenizer.pad_token_id

model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=128)

Embedding(32128, 4096)

In [8]:
weightsEdited = "/bime-munin/xiruod/llama2_SHAC/n200/Weights/set-1355-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.0-lambda2_0.0-added.pth"

In [9]:
# this step cannot be ignored here...
model.load_state_dict(torch.load(weightsEdited, 
                                 # map_location="cuda:0",
                                 map_location="cpu",
                                 # map_location=lambda storage, loc: storage,
                                ))

<All keys matched successfully>

In [10]:

bnb_quantization_config = BnbQuantizationConfig(load_in_8bit=True, llm_int8_threshold = 6)
model = load_and_quantize_model(model, weights_location=weightsEdited, bnb_quantization_config=bnb_quantization_config, device_map = "cpu")

It is not recommended to quantize a loaded model. The model should be instantiated under the `init_empty_weights` context manager.


In [11]:
model

LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(32128, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear8bitLt(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear8bitLt(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNo

In [17]:

model.model.layers[2].self_attn.q_proj.weight

Parameter containing:
Parameter(Int8Params([[-32, -12,  10,  ..., -14, -26, -11],
            [-19,  14, -49,  ..., -47, -23,  35],
            [-18,  41, -33,  ...,  -7, -26,  27],
            ...,
            [-43,  12, -20,  ...,  -6, -27, -33],
            [  1,  -5,   1,  ...,  -1,  -8,   7],
            [ -6, -18,  23,  ...,  56, -42,   2]], device='cuda:0',
           dtype=torch.int8))

In [12]:

model.model.layers[2].self_attn.q_proj.weight

Parameter containing:
Parameter(Int8Params([[-32, -12,  10,  ..., -14, -26, -11],
            [-19,  14, -49,  ..., -47, -23,  35],
            [-18,  41, -33,  ...,  -7, -26,  27],
            ...,
            [-43,  12, -20,  ...,  -6, -27, -33],
            [  1,  -5,   1,  ...,  -1,  -8,   7],
            [ -6, -18,  23,  ...,  56, -42,   2]], device='cuda:0',
           dtype=torch.int8))

In [12]:

model.model.layers[2].self_attn.q_proj.weight

Parameter containing:
Parameter(Int8Params([[-32, -12,  10,  ..., -14, -26, -11],
            [-19,  14, -49,  ..., -47, -23,  35],
            [-18,  41, -33,  ...,  -7, -26,  27],
            ...,
            [-43,  12, -20,  ...,  -6, -27, -33],
            [  1,  -5,   1,  ...,  -1,  -8,   7],
            [ -6, -18,  23,  ...,  56, -42,   2]], device='cuda:0',
           dtype=torch.int8))

In [19]:

model.model.layers[2].self_attn.v_proj.weight

Parameter containing:
Parameter(Int8Params([[-10,   2,  47,  ...,  87, -13,  11],
            [  7,  34, -52,  ..., -23,  13, -38],
            [ -1,  30,  -2,  ..., -32, -58, -59],
            ...,
            [  1, -85, -53,  ..., -36,  -5,   3],
            [-76,  -1,  37,  ..., -16, -22, -12],
            [-28,  38,   3,  ..., -12,  36, -17]], device='cuda:0',
           dtype=torch.int8))

In [13]:

model.model.layers[2].self_attn.v_proj.weight

Parameter containing:
Parameter(Int8Params([[-10,   2,  47,  ...,  87, -13,  11],
            [  7,  34, -52,  ..., -23,  13, -38],
            [ -1,  30,  -2,  ..., -32, -58, -59],
            ...,
            [  1, -85, -53,  ..., -36,  -5,   3],
            [-76,  -1,  37,  ..., -16, -22, -12],
            [-28,  38,   3,  ..., -12,  36, -17]], device='cuda:0',
           dtype=torch.int8))

In [13]:

model.model.layers[2].self_attn.v_proj.weight

Parameter containing:
Parameter(Int8Params([[-10,   2,  47,  ...,  87, -13,  11],
            [  7,  34, -52,  ..., -23,  13, -38],
            [ -1,  30,  -2,  ..., -32, -58, -59],
            ...,
            [  1, -85, -53,  ..., -36,  -5,   3],
            [-76,  -1,  37,  ..., -16, -22, -12],
            [-28,  38,   3,  ..., -12,  36, -17]], device='cuda:0',
           dtype=torch.int8))

In [14]:

model.model.layers[2].self_attn.v_proj.weight

Parameter containing:
Parameter(Int8Params([[-10,   2,  47,  ...,  87, -13,  11],
            [  7,  34, -52,  ..., -23,  13, -38],
            [ -1,  30,  -2,  ..., -32, -58, -59],
            ...,
            [  1, -85, -53,  ..., -36,  -5,   3],
            [-76,  -1,  37,  ..., -16, -22, -12],
            [-28,  38,   3,  ..., -12,  36, -17]], device='cuda:0',
           dtype=torch.int8))

In [14]:

model.model.layers[0].self_attn.q_proj.weight

Parameter containing:
Parameter(Int8Params([[ -7, -17,  -2,  ...,   5,   2,  -4],
            [  9,  -3,   2,  ...,  -6,  -7,   5],
            [ -7,   6,   0,  ...,   3,  10,  -2],
            ...,
            [  1,   6,   0,  ...,   6, -18,   6],
            [ 21,   8,   3,  ..., -27, -13, -10],
            [-11,  -5,   1,  ...,  15,  13,  -7]], device='cuda:0',
           dtype=torch.int8))

In [21]:

model.model.layers[0].self_attn.v_proj.weight

Parameter containing:
Parameter(Int8Params([[  4,  -3,   9,  ...,  25,  -2,  43],
            [-29,  -2, -32,  ..., -44,  52,  19],
            [  8,  45,   5,  ...,  22, -65, -67],
            ...,
            [-29, -27,  48,  ...,  15,  19, -10],
            [ 13,  21,  -5,  ...,  23,  68,   3],
            [  0,  12,  24,  ...,  -3,  -3,   7]], device='cuda:0',
           dtype=torch.int8))

In [15]:

model.model.layers[0].self_attn.v_proj.weight

Parameter containing:
Parameter(Int8Params([[  4,  -3,   9,  ...,  25,  -2,  43],
            [-29,  -2, -32,  ..., -44,  52,  19],
            [  8,  45,   5,  ...,  22, -65, -67],
            ...,
            [-29, -27,  48,  ...,  15,  19, -10],
            [ 13,  21,  -5,  ...,  23,  68,   3],
            [  0,  12,  24,  ...,  -3,  -3,   7]], device='cuda:0',
           dtype=torch.int8))

In [15]:

model.model.layers[0].self_attn.v_proj.weight

Parameter containing:
Parameter(Int8Params([[  4,  -3,   9,  ...,  25,  -2,  43],
            [-29,  -2, -32,  ..., -44,  52,  19],
            [  8,  45,   5,  ...,  22, -65, -67],
            ...,
            [-29, -27,  48,  ...,  15,  19, -10],
            [ 13,  21,  -5,  ...,  23,  68,   3],
            [  0,  12,  24,  ...,  -3,  -3,   7]], device='cuda:0',
           dtype=torch.int8))

In [17]:
import numpy as np

In [32]:
from collections import Counter
from sklearn.datasets import make_classification
# define dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, n_redundant=5, n_classes=6, random_state=1)
# summarize the dataset
print(X.shape, y.shape)
print(Counter(y))

(1000, 10) (1000,)
Counter({3: 169, 2: 168, 0: 167, 1: 166, 4: 165, 5: 165})


In [33]:
from sklearn.linear_model import LogisticRegression

In [34]:
penalty = "l2"
solver = "lbfgs"

In [35]:
clf_z  = LogisticRegression(penalty = penalty, C = 10, max_iter = 1000, class_weight = None, solver= solver, multi_class="multinomial")

In [36]:
clf_z.fit(X, y)

LogisticRegression(C=10, max_iter=1000, multi_class='multinomial')

In [37]:
clf_z.predict_proba(X)

array([[0.01693785, 0.12171967, 0.02483324, 0.28929718, 0.25930334,
        0.28790874],
       [0.02240364, 0.16339595, 0.260147  , 0.3259346 , 0.09871595,
        0.12940286],
       [0.09952638, 0.05720693, 0.43138007, 0.16099103, 0.20375689,
        0.0471387 ],
       ...,
       [0.15828058, 0.09461486, 0.20948857, 0.06958397, 0.26848726,
        0.19954476],
       [0.04919271, 0.28804542, 0.11013555, 0.22506083, 0.16211607,
        0.16544941],
       [0.13320239, 0.14599467, 0.27025514, 0.13573652, 0.20908984,
        0.10572144]])

In [38]:
tmp = np.array([[0,1,2],[2,3,4]])

In [42]:
tmp

array([[0, 1, 2],
       [2, 3, 4]])

In [40]:
(10-tmp) * tmp

array([[ 0,  9, 16],
       [16, 21, 24]])

In [41]:
(10-tmp) 

array([[10,  9,  8],
       [ 8,  7,  6]])

In [43]:
tmp = clf_z.predict_proba(X)[:,1]

In [50]:
np.stack([tmp,1-tmp], axis=1)

array([[0.12171967, 0.87828033],
       [0.16339595, 0.83660405],
       [0.05720693, 0.94279307],
       ...,
       [0.09461486, 0.90538514],
       [0.28804542, 0.71195458],
       [0.14599467, 0.85400533]])